# Triviality of Inhomogeneous Deformations of $\mathfrak{osp}(1|2n)$

This notebook demonstrates the computational verification of the main theorem:

> **Theorem.** For $n \geq 2$, the $\gamma$-deformation of $B(0,n) = \mathfrak{osp}(1|2n)$ is trivial
> if and only if all deformation parameters vanish.

We verify this using:
1. **Certificate verification**: Algebraic proof that $c^\top A = 0$ and $c^\top L \neq 0$
2. **Triviality checking**: Numerical verification of $\delta f = \gamma$
3. **Dimension formula**: $\dim C^1_\mu = 6n - 2$ for weight sectors $\mu = \pm e_j$

For full details, see the [paper](https://arxiv.org/) and the [repository](https://github.com/ritsumei-aoi/osp-triviality).

## Setup


In [1]:
import sys
import json
import os
from fractions import Fraction
from pathlib import Path
from collections import defaultdict

# Set root to the repository root (parent of notebooks/)
root = Path(os.path.abspath("")).resolve().parent
if (root / "notebooks").exists():
    pass  # already at repo root
elif root.name == "notebooks":
    root = root.parent

# Ensure src/ and scripts/ are on the path
if str(root / 'src') not in sys.path:
    sys.path.insert(0, str(root / 'src'))
if str(root / 'scripts') not in sys.path:
    sys.path.insert(0, str(root / 'scripts'))

# Change working directory to repo root for data file access
os.chdir(root)

from verify_certificates_algebraic import (
    load_structure_constants, load_gamma_constants,
    get_family_I_certificate, get_family_II_certificate,
    get_family_III_certificate_minus, get_family_III_certificate_plus,
    verify_certificate, verify_all_certificates
)
from oscillator_lie_superalgebras.triviality_checker import check_triviality

print(f'Repository root: {root}')
print('All modules loaded successfully.')


Repository root: /Users/aoi/notebook/git/osp-triviality
All modules loaded successfully.


## 1. Algebra Structure of $\mathfrak{osp}(1|2n)$

The oscillator Lie superalgebra $B(0,n) = \mathfrak{osp}(1|2n)$ has:
- Even part $\mathfrak{g}_{\bar{0}} \cong \mathfrak{sp}(2n)$ with dimension $2n^2 + n$
- Odd part $\mathfrak{g}_{\bar{1}}$ with dimension $2n$

The even part is spanned by Cartan elements $H_j$ and root vectors $E_{\alpha}^\pm$.
The odd part is spanned by $E_{\delta_j}^\pm$ ($j = 1, \ldots, n$).

Let's examine the structure for $n = 2$ and $n = 3$.

In [2]:
print(root)

/Users/aoi/notebook/git/osp-triviality


In [3]:
os.chdir(root)
for n_val in [2, 3]:
    with open(f'data/algebra_structures/B_0_{n_val}_structure.json') as f:
        alg_data = json.load(f)

    even = alg_data['basis']['even']
    odd = alg_data['basis']['odd']
    print(f'B(0,{n_val}) = osp(1|{2*n_val})')
    print(f'  Even basis ({len(even)} elements): {even}')
    print(f'  Odd basis ({len(odd)} elements): {odd}')
    print(f'  Total dimension: {len(even) + len(odd)}')
    print(f'  Expected (2n²+n | 2n) = ({2*n_val**2+n_val} | {2*n_val})')
    print()

B(0,2) = osp(1|4)
  Even basis (10 elements): ['H_1', 'H_2', 'E_2del1_p', 'E_2del2_p', 'E_del1_del2_pp', 'E_2del1_m', 'E_2del2_m', 'E_del1_del2_mm', 'E_del1_del2_pm', 'E_del1_del2_mp']
  Odd basis (4 elements): ['E_del1_p', 'E_del2_p', 'E_del1_m', 'E_del2_m']
  Total dimension: 14
  Expected (2n²+n | 2n) = (10 | 4)

B(0,3) = osp(1|6)
  Even basis (21 elements): ['H_1', 'H_2', 'H_3', 'E_2del1_p', 'E_2del2_p', 'E_2del3_p', 'E_del1_del2_pp', 'E_del1_del3_pp', 'E_del2_del3_pp', 'E_2del1_m', 'E_2del2_m', 'E_2del3_m', 'E_del1_del2_mm', 'E_del1_del3_mm', 'E_del2_del3_mm', 'E_del1_del2_pm', 'E_del1_del2_mp', 'E_del1_del3_pm', 'E_del1_del3_mp', 'E_del2_del3_pm', 'E_del2_del3_mp']
  Odd basis (6 elements): ['E_del1_p', 'E_del2_p', 'E_del3_p', 'E_del1_m', 'E_del2_m', 'E_del3_m']
  Total dimension: 27
  Expected (2n²+n | 2n) = (21 | 6)



## 2. Triviality Check

The coboundary equation $\delta f = \gamma$ decomposes by weight into:
$$A_\mu \cdot \mathbf{f}_\mu = L_\mu \cdot \mathbf{gb}_\mu$$

where $A_\mu$ is the coboundary matrix, $L_\mu$ is the deformation matrix,
and $\mathbf{gb}_\mu$ is the vector of deformation parameters in weight sector $\mu$.

### Exceptional case: $n = 1$ (always trivial)

For $B(0,1) = \mathfrak{osp}(1|2)$, the coboundary equation always has a solution,
so every deformation is trivial.

In [4]:
print('=== B(0,1) = osp(1|2): Exceptional case ===')
for gb_values in [[1.0, -0.5], [0.3, 0.7], [2.0, -1.0]]:
    result = check_triviality(n=1, B=gb_values)
    status = '✓ TRIVIAL' if result['is_trivial'] else '✗ NON-TRIVIAL'
    print(f'  gb = {gb_values} → {status} (residual: {result["residual_norm"]:.2e})')

print('\nAll deformations are trivial regardless of parameter values.')

=== B(0,1) = osp(1|2): Exceptional case ===
  gb = [1.0, -0.5] → ✓ TRIVIAL (residual: 2.63e-15)
  gb = [0.3, 0.7] → ✓ TRIVIAL (residual: 1.85e-15)
  gb = [2.0, -1.0] → ✓ TRIVIAL (residual: 5.26e-15)

All deformations are trivial regardless of parameter values.


### General case: $n \geq 2$ (trivial iff $\mathbf{gb} = 0$)

For $n \geq 2$, the deformation is trivial only when all parameters vanish.

In [5]:
print('=== B(0,2) = osp(1|4): General case ===')
test_cases = [
    ([0, 0, 0, 0], 'gb = 0'),
    ([1, 0, 0, 0], 'gb₁⁺ ≠ 0'),
    ([0, 1, 0, 0], 'gb₂⁺ ≠ 0'),
    ([0, 0, 1, 0], 'gb₁⁻ ≠ 0'),
    ([0, 0, 0, 1], 'gb₂⁻ ≠ 0'),
    ([0.5, -0.3, 1.0, 0.7], 'all nonzero'),
]

for gb_values, label in test_cases:
    result = check_triviality(n=2, B=gb_values)
    status = '✓ TRIVIAL' if result['is_trivial'] else '✗ NON-TRIVIAL'
    print(f'  {label:20s} → {status} (residual: {result["residual_norm"]:.2e})')

print('\n→ Only gb = 0 gives a trivial deformation.')

=== B(0,2) = osp(1|4): General case ===
  gb = 0               → ✓ TRIVIAL (residual: 0.00e+00)
  gb₁⁺ ≠ 0             → ✗ NON-TRIVIAL (residual: 1.86e+00)
  gb₂⁺ ≠ 0             → ✗ NON-TRIVIAL (residual: 1.86e+00)


  gb₁⁻ ≠ 0             → ✗ NON-TRIVIAL (residual: 1.31e+00)


  gb₂⁻ ≠ 0             → ✗ NON-TRIVIAL (residual: 1.31e+00)
  all nonzero          → ✗ NON-TRIVIAL (residual: 1.93e+00)

→ Only gb = 0 gives a trivial deformation.


## 3. Certificate Verification

A **certificate** is a vector $c$ in the left null space of $A_\mu$ such that $c^\top L_\mu \neq 0$.
Its existence proves that $\delta f = \gamma$ has no solution when the corresponding
deformation parameter is nonzero.

Three certificate families cover all $2n$ deformation parameters:

| Family | Parameters covered | Index range |
|--------|-------------------|-------------|
| I | $\mathrm{gb}_{a_0, b_j^+}$ | $j = 1, \ldots, n-1$ |
| II | $\mathrm{gb}_{a_0, b_j^-}$ | $j = 1, \ldots, n-1$ |
| III | $\mathrm{gb}_{a_0, b_n^\pm}$ | (two certificates) |

### Full verification for $n = 2, 3, 4$

In [6]:
for n_val in [2, 3, 4]:
    verify_all_certificates(0, n_val)


  B(0,2) = osp(1|4) — Algebraic certificate verification

  ✓ Family I: b_1^+ (j=1)
    c^T A = 0: True
    c^T L = {'gb_a0_b1p': Fraction(4, 1)}  (target gb: gb_a0_b1p = 4)

  ✓ Family II: b_1^- (j=1)
    c^T A = 0: True
    c^T L = {'gb_a0_b1m': Fraction(2, 1)}  (target gb: gb_a0_b1m = 2)

  ✓ Family III: b_2^- (n=2)
    c^T A = 0: True
    c^T L = {'gb_a0_b2m': Fraction(1, 1)}  (target gb: gb_a0_b2m = 1)

  ✓ Family III: b_2^+ (n=2)
    c^T A = 0: True
    c^T L = {'gb_a0_b2p': Fraction(1, 1)}  (target gb: gb_a0_b2p = 1)

  Summary: 4/4 certificates verified (ALL PASS)

  c^T L values:
    Family I: b_1^+ (j=1): 4
    Family II: b_1^- (j=1): 2
    Family III: b_2^- (n=2): 1
    Family III: b_2^+ (n=2): 1

  B(0,3) = osp(1|6) — Algebraic certificate verification

  ✓ Family I: b_1^+ (j=1)
    c^T A = 0: True
    c^T L = {'gb_a0_b1p': Fraction(4, 1)}  (target gb: gb_a0_b1p = 4)

  ✓ Family I: b_2^+ (j=2)
    c^T A = 0: True
    c^T L = {'gb_a0_b2p': Fraction(4, 1)}  (target gb: gb_a0

## 4. Detailed Certificate Inspection

Let's examine individual certificates for $n = 2$ to see the algebraic structure.

Each certificate is a linear combination of coboundary evaluations
$(\delta f)(X, Y)|_Z$ that lies in $\ker A_\mu^\top$ but not in $\ker L_\mu^\top$.

In [7]:
n = 2
basis, parity, brackets = load_structure_constants(0, n)
gb_vars, gamma = load_gamma_constants(0, n)

print(f'B(0,{n}): {len(basis)} basis elements, {len(gb_vars)} deformation parameters')
print(f'Parameters: {gb_vars}')
print()

# Family I, j=1
cert_I = get_family_I_certificate(1, n)
print('=== Family I certificate (j=1) ===')
print('Components:')
for X, Y, Z, coeff in cert_I:
    print(f'  {float(coeff):>3} · (δf)({X}, {Y})|_{Z}')

ok, ctL = verify_certificate(cert_I, basis, parity, brackets, gamma, gb_vars, 'Family I, j=1')
print(f'\nVerification:')
print(f'  c^T A = 0: {ok}')
print(f'  c^T L = {dict(ctL)}')
print(f'  → Forces gb_a0_b1p = 0 ✓')

B(0,2): 14 basis elements, 4 deformation parameters
Parameters: ['gb_a0_b1p', 'gb_a0_b1m', 'gb_a0_b2p', 'gb_a0_b2m']

=== Family I certificate (j=1) ===
Components:
  1.0 · (δf)(H_1, E_2del1_p)|_E_del1_p
  2.0 · (δf)(H_1, E_del1_p)|_H_2
  -4.0 · (δf)(H_1, E_del1_m)|_E_2del1_m
  1.0 · (δf)(E_2del1_p, E_del1_m)|_H_2

Verification:
  c^T A = 0: True
  c^T L = {'gb_a0_b1p': Fraction(4, 1)}
  → Forces gb_a0_b1p = 0 ✓


In [8]:
# Family III certificates
for label, getter in [('III⁺', get_family_III_certificate_plus),
                       ('III⁻', get_family_III_certificate_minus)]:
    cert = getter(n)
    print(f'=== Family {label} certificate (n={n}) ===')
    print('Components:')
    for X, Y, Z, coeff in cert:
        print(f'  {float(coeff):>3} · (δf)({X}, {Y})|_{Z}')

    ok, ctL = verify_certificate(cert, basis, parity, brackets, gamma, gb_vars, f'Family {label}')
    print(f'\nVerification:')
    print(f'  c^T A = 0: {ok}')
    print(f'  c^T L = {dict(ctL)}')
    print()

=== Family III⁺ certificate (n=2) ===
Components:
  -1.0 · (δf)(H_1, E_del1_del2_pp)|_E_del1_p
  -1.0 · (δf)(H_1, E_del2_p)|_H_2
  2.0 · (δf)(H_1, E_del1_m)|_E_del1_del2_mm
  1.0 · (δf)(E_del1_del2_pp, E_del1_m)|_H_2

Verification:
  c^T A = 0: True
  c^T L = {'gb_a0_b2p': Fraction(1, 1)}

=== Family III⁻ certificate (n=2) ===
Components:
  1.0 · (δf)(H_1, E_del1_del2_mm)|_E_del1_m
  2.0 · (δf)(H_1, E_del1_p)|_E_del1_del2_pp
  -1.0 · (δf)(H_1, E_del2_m)|_H_2
  1.0 · (δf)(E_del1_del2_mm, E_del1_p)|_H_2

Verification:
  c^T A = 0: True
  c^T L = {'gb_a0_b2m': Fraction(1, 1)}



## 5. Dimension Formula

The dimension of the $f$-variable space in each weight sector $\mu = \pm e_j$ is:
$$\dim C^1_\mu = 6n - 2$$

This is derived from an 8-category decomposition of the contributing
basis pairs $(\text{src}, \text{dst})$ with mixed parity and weight difference $\mu$.

In [9]:
from verify_dim_formula import count_by_formula, get_fvar_sectors, load_structure, get_gb_weights

print('Dimension formula verification: dim C^1_μ = 6n - 2')
print(f'{"n":>3} {"Formula":>10} {"Computed":>10} {"Match":>6}')
print('-' * 35)

for n_val in range(1, 6):
    formula_count, _, _ = count_by_formula(n_val)
    expected = 6 * n_val - 2
    match = '✓' if formula_count == expected else '✗'
    print(f'{n_val:>3} {expected:>10} {formula_count:>10} {match:>6}')

Dimension formula verification: dim C^1_μ = 6n - 2
  n    Formula   Computed  Match
-----------------------------------
  1          4          4      ✓
  2         10         10      ✓
  3         16         16      ✓
  4         22         22      ✓
  5         28         28      ✓


In [10]:
# Cross-check with actual structure data for n=2,3
print('Cross-check with actual algebra structure data:')
print()

for n_val in [2, 3]:
    structure = load_structure(0, n_val)
    if structure is None:
        continue
    sectors = get_fvar_sectors(structure, n_val)
    gb_weights = get_gb_weights(n_val)
    
    print(f'B(0,{n_val}): gb weight sectors')
    for mu in gb_weights:
        count = len(sectors.get(mu, []))
        expected = 6 * n_val - 2
        match = '✓' if count == expected else '✗'
        print(f'  μ = {mu}: dim = {count} (expected {expected}) {match}')
    print()

Cross-check with actual algebra structure data:

B(0,2): gb weight sectors
  μ = (-1, 0): dim = 10 (expected 10) ✓
  μ = (-1, 1): dim = 10 (expected 10) ✓
  μ = (1, -1): dim = 10 (expected 10) ✓
  μ = (1, 0): dim = 10 (expected 10) ✓

B(0,3): gb weight sectors
  μ = (-1, 0, 0): dim = 16 (expected 16) ✓
  μ = (-1, 1, 0): dim = 16 (expected 16) ✓
  μ = (0, -1, 1): dim = 16 (expected 16) ✓
  μ = (0, 1, -1): dim = 16 (expected 16) ✓
  μ = (1, -1, 0): dim = 16 (expected 16) ✓
  μ = (1, 0, 0): dim = 16 (expected 16) ✓



## 6. 4-Layer JSON Schema Pipeline

The computation pipeline is managed using a **4-layer JSON schema**:

| Schema | File Pattern | Content |
|--------|-------------|--------|
| 1 | `B_m_n_structure.json` | Lie superalgebra structure (basis, parity, brackets) |
| 2 | `B_m_n_gamma.json` | Symbolic $\mathrm{gb}$ variables in $\gamma$ components |
| 3 | (computed) | Numerical $\mathrm{gb}$ values → concrete $\gamma$ brackets |
| 4 | (computed) | $f$ map → $\delta f$ coboundary components |

**Forward Flow** (Parameters → Deformation):
$$\text{Schema 1} \;\longrightarrow\; \text{Schema 2 (symbolic)} \;\longrightarrow\; \text{Schema 3 (numerical)}$$

**Backward Flow** (Map → Coboundary):
$$\text{Schema 1} \;\longrightarrow\; \text{Schema 4 ($f \to \delta f$)}$$

Triviality is determined by checking whether $\gamma$ (Schema 3) lies in the image of $\delta$ (Schema 4).

### Schema 1: Algebra Structure

Contains the Lie superalgebra structure constants, basis elements, and parity assignments.

In [11]:
import json, os
os.chdir(root)

with open('data/algebra_structures/B_0_2_structure.json') as f:
    schema1 = json.load(f)

print('Schema 1: B(0,2) = osp(1|4)')
print(f'  schema_version: {schema1["schema_version"]}')
print(f'  Even basis: {schema1["basis"]["even"]}')
print(f'  Odd basis:  {schema1["basis"]["odd"]}')

brackets = schema1['structure_constants']['brackets']
print(f'  # non-zero brackets: {len(brackets)}')
print()
print('Sample bracket entries:')
for pair_key in list(brackets)[:6]:
    left, right = pair_key.split(',')
    terms = [f'{c}·{b}' for b, c in brackets[pair_key].items()]
    print(f'  [{left}, {right}] = {" + ".join(terms)}')

Schema 1: B(0,2) = osp(1|4)
  schema_version: 5.0
  Even basis: ['H_1', 'H_2', 'E_2del1_p', 'E_2del2_p', 'E_del1_del2_pp', 'E_2del1_m', 'E_2del2_m', 'E_del1_del2_mm', 'E_del1_del2_pm', 'E_del1_del2_mp']
  Odd basis:  ['E_del1_p', 'E_del2_p', 'E_del1_m', 'E_del2_m']
  # non-zero brackets: 108

Sample bracket entries:
  [H_1, E_2del1_p] = 2·E_2del1_p
  [H_1, E_2del1_m] = -2·E_2del1_m
  [H_1, E_2del2_p] = -2·E_2del2_p
  [H_1, E_2del2_m] = 2·E_2del2_m
  [H_1, E_del1_del2_pm] = 2·E_del1_del2_pm
  [H_1, E_del1_del2_mp] = -2·E_del1_del2_mp


### Schema 2: Symbolic Gamma Structure

Contains the $\gamma$-deformation brackets with symbolic $\mathrm{gb}$ parameters.
The exchange relations are: $[b_j, a_i] = -\mathrm{gb}[i][j] \cdot \kappa$.

In [12]:
with open('data/gamma_structures/B_0_2_gamma.json') as f:
    schema2 = json.load(f)

inhom = schema2['inhomogeneous_deformation']
gb_matrix = inhom['inhomogeneous_matrix']

print('Schema 2: B(0,2) symbolic gamma')
print(f'  schema_version: {schema2["schema_version"]}')
print(f'  Exchange relation: {inhom["exchange_relation"]}')
print(f'  gb matrix ({gb_matrix["shape"][0]}×{gb_matrix["shape"][1]}):')
for row in gb_matrix['matrix']:
    print(f'    {row}')
print()
print('Sample gamma bracket entries (symbolic):')
gamma_brackets = schema2['gamma_matrix']['brackets']
count = 0
for pair_key in sorted(gamma_brackets):
    if count >= 4:
        break
    components = gamma_brackets[pair_key]
    terms = []
    for basis_el, gb_dict in components.items():
        for gb_var, coeff in gb_dict.items():
            terms.append(f'{coeff}·{gb_var}·{basis_el}')
    if terms:
        left, right = pair_key.split(',')
        print(f'  γ({left}, {right}) = {" + ".join(terms)}')
        count += 1

Schema 2: B(0,2) symbolic gamma
  schema_version: 5.0
  Exchange relation: [b_j, a_i] = G[b_j][a_i]·κ = -gb[i][j]·κ
  gb matrix (1×4):
    ['gb_a0_b1p', 'gb_a0_b1m', 'gb_a0_b2p', 'gb_a0_b2m']

Sample gamma bracket entries (symbolic):
  γ(E_2del1_m, E_del1_m) = -2·gb_a0_b1m·E_2del1_m
  γ(E_2del1_m, E_del1_p) = -2·gb_a0_b1m·H_1
  γ(E_2del1_m, E_del2_m) = -2·gb_a0_b1m·E_del1_del2_mm
  γ(E_2del1_m, E_del2_p) = -2·gb_a0_b1m·E_del1_del2_mp


### Forward Flow: Schema 2 → Schema 3

Substituting concrete $\mathrm{gb}$ values into the symbolic gamma yields numerical brackets.
This is the **evaluated** deformation for specific parameter choices.

In [13]:
from oscillator_lie_superalgebras.gamma_from_gb import compute_gamma_from_gb

# B(0,2): gb is 1×4 matrix (1 fermion × 4 bosons)
# Activate only the first parameter: gb_a0_b1p = 1
gb_example = [[1, 0, 0, 0]]

gamma_numerical = compute_gamma_from_gb(
    'data/gamma_structures/B_0_2_gamma.json', gb_example
)

print('Schema 3 (Forward Flow): Numerical gamma for B(0,2)')
print(f'  gb matrix: {gb_example}')
print(f'  Non-zero gamma entries: {len(gamma_numerical)}')
print()
for (g1, g2), coeffs in sorted(gamma_numerical.items()):
    terms = [f'{v.real:+g}·{k}' for k, v in sorted(coeffs.items()) if abs(v) > 1e-12]
    if terms:
        print(f'  [{g1}, {g2}] = {", ".join(terms)}')

Schema 3 (Forward Flow): Numerical gamma for B(0,2)
  gb matrix: [[1, 0, 0, 0]]
  Non-zero gamma entries: 20

  [E_2del1_p, E_del1_m] = -2·H_1
  [E_2del1_p, E_del1_p] = -2·E_2del1_p
  [E_2del1_p, E_del2_m] = -2·E_del1_del2_pm
  [E_2del1_p, E_del2_p] = -2·E_del1_del2_pp
  [E_del1_del2_pm, E_del1_m] = -1·E_del1_del2_mm
  [E_del1_del2_pm, E_del1_p] = -1·E_del1_del2_pm
  [E_del1_del2_pm, E_del2_m] = -1·E_2del2_m
  [E_del1_del2_pm, E_del2_p] = -1·H_2
  [E_del1_del2_pp, E_del1_m] = -1·E_del1_del2_mp
  [E_del1_del2_pp, E_del1_p] = -1·E_del1_del2_pp
  [E_del1_del2_pp, E_del2_m] = -1·H_2
  [E_del1_del2_pp, E_del2_p] = -1·E_2del2_p
  [E_del1_p, E_del1_m] = +1·E_del1_m
  [E_del1_p, E_del1_p] = +2·E_del1_p
  [E_del1_p, E_del2_m] = +1·E_del2_m
  [E_del1_p, E_del2_p] = +1·E_del2_p
  [H_1, E_del1_m] = -1·E_2del1_m
  [H_1, E_del1_p] = -1·H_1
  [H_1, E_del2_m] = -1·E_del1_del2_mm
  [H_1, E_del2_p] = -1·E_del1_del2_mp


### Backward Flow: Schema 1 → Schema 4

Given a linear map $f: \mathfrak{g} \to \mathfrak{g}$ (odd, degree-shifting),
the coboundary $\delta f$ is computed from the structure constants (Schema 1).

For $B(0,1)$, we explicitly construct an $f$ such that $\delta f = \gamma$,
proving triviality for all $\mathrm{gb}$ values.

In [14]:
import numpy as np
import sympy as sp
from oscillator_lie_superalgebras.trivial_subspace import (
    build_coboundary_matrix, build_gamma_linear_map
)

M, N = 0, 1  # B(0,1) = osp(1|2)

# Build coboundary matrix A and deformation matrix L
A_num, basis, parity = build_coboundary_matrix(M, N)
L_num, gamma_index, gb_vars, _, _ = build_gamma_linear_map(M, N)

print('Schema 4 (Backward Flow): Coboundary for B(0,1)')
print(f'  Basis: {basis}')
print(f'  gb parameters: {gb_vars}')
print(f'  Coboundary matrix A: {A_num.shape}, rank = {np.linalg.matrix_rank(A_num)}')
print(f'  Deformation matrix L: {L_num.shape}, rank = {np.linalg.matrix_rank(L_num)}')
print()

# Solve A·f = L·gb exactly over Q (SymPy rational arithmetic)
A_sym = sp.Matrix(A_num.real.tolist()).applyfunc(sp.nsimplify)
L_sym = sp.Matrix(L_num.real.tolist()).applyfunc(sp.nsimplify)

# f-variable index: (src, dst) pairs with opposite parity
f_pairs = [(s, d) for s in basis for d in basis if parity[s] != parity[d]]

print('Explicit f map (particular solution, free variables set to 0):')
for k, gb_name in enumerate(gb_vars):
    rhs = L_sym.col(k)
    sol, free = A_sym.gauss_jordan_solve(rhs)
    particular = sol.subs({s: 0 for s in free})
    residual = A_sym * particular - rhs
    print(f'\n  Direction: {gb_name} (residual = {float(residual.norm()):.0e})')
    for idx, (src, dst) in enumerate(f_pairs):
        val = particular[idx]
        if val != 0:
            print(f'    f({src}) += {val} · {dst}')

Schema 4 (Backward Flow): Coboundary for B(0,1)
  Basis: ['H_1', 'E_2del1_p', 'E_2del1_m', 'E_del1_p', 'E_del1_m']
  gb parameters: ['gb_a0_b1p', 'gb_a0_b1m']
  Coboundary matrix A: (60, 12), rank = 10
  Deformation matrix L: (60, 2), rank = 2

Explicit f map (particular solution, free variables set to 0):

  Direction: gb_a0_b1p (residual = 0e+00)
    f(H_1) += -1 · E_del1_m
    f(E_2del1_p) += -2 · E_del1_p

  Direction: gb_a0_b1m (residual = 0e+00)
    f(H_1) += -1 · E_del1_p
    f(E_2del1_m) += -2 · E_del1_m


### Pipeline Verification

The 4-layer pipeline ensures:
1. **Schema 1** provides the algebraic structure (independently verifiable)
2. **Schema 2** encodes the parametric deformation (symbolic, reusable)
3. **Schema 3** instantiates specific deformations for testing
4. **Schema 4** constructs coboundaries to check triviality

The triviality question reduces to: does $\gamma$ (Schema 3) lie in the image of $\delta$ (Schema 4)?

## Summary

| Property | $n = 1$ | $n \geq 2$ |
|----------|---------|------------|
| Triviality | Always trivial | Trivial iff $\mathrm{gb} = 0$ |
| Certificates | Not needed | $2n$ certificates (Families I, II, III) |
| $\dim C^1_\mu$ | $4$ | $6n - 2$ |
| Rank condition | $\operatorname{rank}([A\|L]) = \operatorname{rank}(A)$ | $\operatorname{rank}([A\|L]) > \operatorname{rank}(A)$ |

### Explore further

```python
# Check triviality for B(0,3) with specific parameters
result = check_triviality(n=3, B=[1, 0, 0, 0, 0, 0])

# Verify certificates for larger n (n=5 takes a few seconds)
verify_all_certificates(0, 5)
```

See the [scripts/](../scripts/) directory for the full implementation.